### Environment Setup and Model Definition for Hypergraph Partitioning
This section contains the environment configuration, helper functions, and model definition for the hypergraph partitioning experiments, including random seed control and the DirectProbModel network.


First of all, ensure you can import our main code `src`:

In [3]:
import sys
from pathlib import Path
current = Path.cwd()
root_path = current.parents[3]  
sys.path.append(str(root_path))
import random
import numpy as np
import torch
import torch.nn as nn

from src import loss_partitioning_onehot_qubo
from src import from_file_to_graph, init, get_device, run_qubo, from_file_to_hypergraph, run_pubo
from src import Layer, LayerType, Datasets

/home/guohao/miniconda3/envs/kgrouping/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Set random seeds for reproducibility:

In [1]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

### Define the DirectProbModel:

In [4]:
class DirectProbModel(nn.Module):
    def __init__(self, num_nodes, num_classes):
        super().__init__()
        self.h = nn.Parameter(torch.randn(num_nodes, num_classes))

    def forward(self, X, *args, **kwargs):
        p = torch.softmax(self.h, dim=-1)
        return (p,)

### Main Experiment Script for Hypergraph Partitioning
This section contains the main loop for the hypergraph partitioning task, loading hypergraph data, running run_pubo for optimization, and recording results across multiple runs.

In [ ]:
init(cuda_index=1, reproducibility=False)
results = []
for run_idx in range(9, 11):
    set_seed(run_idx * 10)
    data_path = Datasets.Hypergraph_high.path
    hg = from_file_to_hypergraph(data_path, True).to(get_device())
    net = DirectProbModel(hg.num_v, 2).to(get_device())
    gini_cof_lambda = lambda e, n: (-800 + 0.25 * e) / 1000
    obj_cof_lambda = lambda e, n: e / 900
    cons_cof_lambda = lambda e, n: e / 2000 +e*0.001
    loss, outs, res = run_pubo(
        type="partitioning", net=net, X=torch.randn(1,1).to(get_device()),
        hypergraph=hg, num_epochs=5000,
        lr=0.5, 
        opt='AdamW', 
        gini_cof_lambda=gini_cof_lambda,
        obj_cof_lambda=obj_cof_lambda,
        cons_cof_lambda=cons_cof_lambda,
        clip_grad=False, 
        evaluate=True
    )
    results.append([run_idx, res["cuts"], res["blce"]])

[INFO] Using CUDA device: NVIDIA GeForce RTX 3090 (Index: 1)
[INFO] INFO: 0 loops found
[INFO] Hypergraph to graph: 327 vertices 5818 graph edges


Hypergraph(num_v=327, num_e=7818)


 42%|████▏     | 2078/5000 [00:02<00:03, 758.04it/s]

Epoch: 2000 | partitioning Loss: 772.01 | balance Loss: 12.50 | gini Loss: 0.00


 82%|████████▏ | 4103/5000 [00:05<00:01, 791.75it/s]

...

Epoch: 2000 | partitioning Loss: 772.01 | balance Loss: 12.50 | gini Loss: 0.00


 83%|████████▎ | 4141/5000 [00:05<00:01, 745.56it/s]

Epoch: 4000 | partitioning Loss: 772.00 | balance Loss: 12.50 | gini Loss: 0.00


100%|██████████| 5000/5000 [00:06<00:00, 763.85it/s] 


+------------[Evaluation Result]------------+
Cuts: 772
Blce: [166 161]
Not converged nodes: 0
+-------------------------------------------+
